# Module 05 — Notebook 2 Solutions: Distributions and Scatter

In [ ]:
import sys
sys.path.insert(0, "../../../")
from src.checks import check_equal, check_type, check_approx, check_length, check_contains
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import json
from pathlib import Path
%matplotlib inline
sns.set_theme(style="whitegrid")
df = pd.read_csv(Path("../../../data/synthetic/evaluation_results.csv"))
with open(Path("../../../data/synthetic/model_outputs.json")) as f:
    outputs_df = pd.DataFrame(json.load(f))
outputs_df["response_length"] = outputs_df["response"].str.len()

## Exercise 1 Solution

In [ ]:
score_stats = df.groupby("model")["score"].agg(["mean", "std", "min", "max"])
# pandas .std() uses ddof=1 (sample std) by default
a2_std = round(float(score_stats.loc["model-a-v2", "std"]), 3)

fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(data=df, x="score", hue="model", bins=6, alpha=0.6, ax=ax)
ax.set_title("Score Distribution by Model")
plt.tight_layout()
plt.show()

In [ ]:
check_type(score_stats, pd.DataFrame, "score_stats is a DataFrame")
check_contains(list(score_stats.columns), "std", "score_stats has std column")
check_approx(a2_std, 0.059, 1e-3, "a2_std")
check_equal(isinstance(fig, plt.Figure), True, "fig is a Figure")

## Exercise 2 Solution

In [ ]:
# Median per task — pandas median uses the middle value of sorted data
hardest_task = df.groupby("task")["score"].median().idxmin()

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x="task", y="score", palette="muted", width=0.5, ax=ax)
sns.stripplot(data=df, x="task", y="score", color="black", size=7, alpha=0.7, jitter=False, ax=ax)
ax.tick_params(axis="x", rotation=30)
ax.set_title("Score Distribution by Task")
plt.tight_layout()
plt.show()

In [ ]:
check_equal(hardest_task, "creative_writing", "hardest task by median is creative_writing")
check_equal(isinstance(fig, plt.Figure), True, "fig is a Figure")

## Exercise 3 Solution

In [ ]:
mean_flagged_length = round(float(outputs_df[outputs_df["flagged"]]["response_length"].mean()), 1)
mean_clean_length   = round(float(outputs_df[~outputs_df["flagged"]]["response_length"].mean()), 1)

fig, ax = plt.subplots(figsize=(9, 5))
# Scatter: response_length on x, row index on y, colored by flagged
for flagged, color, label in [(True, "crimson", "Flagged"), (False, "steelblue", "Clean")]:
    subset = outputs_df[outputs_df["flagged"] == flagged]
    ax.scatter(subset["response_length"], subset.index, color=color, label=label, s=70, alpha=0.8)
ax.axvline(mean_flagged_length, color="crimson", linestyle="--", alpha=0.6, label=f"Flagged mean ({mean_flagged_length})")
ax.axvline(mean_clean_length, color="steelblue", linestyle="--", alpha=0.6, label=f"Clean mean ({mean_clean_length})")
ax.set_xlabel("Response Length (characters)")
ax.set_title("Response Length vs. Flag Status")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
check_approx(mean_flagged_length, 64.7, 0.2, "mean_flagged_length")
check_approx(mean_clean_length, 94.5, 0.2, "mean_clean_length")
check_equal(mean_flagged_length < mean_clean_length, True, "flagged responses are shorter")